# 03 — Dim: shortcut map (Fabric REST → Delta)

For every monitored workspace, lists every lakehouse via `GET /v1/workspaces/{ws}/items?type=Lakehouse`, calls `GET /v1/workspaces/{ws}/items/{item}/shortcuts` for each, and **overwrites** `dim_shortcut_map` with the current snapshot of all shortcuts.

**Auth:** `notebookutils.credentials.getToken('pbi')` — runs as the user / service principal executing the notebook. That identity needs `Contributor` (or higher) on each monitored workspace.

In [ ]:
import json, os
from pyspark.sql import functions as F

# When run inside Fabric, the notebook resource folder contains config.json.
# Fabric exposes notebook-attached files via mssparkutils / notebookutils.
try:
    import notebookutils  # type: ignore
    cfg_path = notebookutils.nbResPath + '/builtin/config.json'
    if not os.path.exists(cfg_path):
        # Fallback: lakehouse Files/config.json
        cfg_path = '/lakehouse/default/Files/config.json'
except Exception:
    cfg_path = './config.json'

with open(cfg_path, 'r', encoding='utf-8') as f:
    CFG = json.load(f)

OBS_WS  = CFG['observability_workspace_name']
OBS_LH  = CFG['observability_lakehouse_name']
TBL     = CFG['tables']
API     = CFG['fabric_api']
# monitored_workspaces is a list of workspace display names (strings).
# Backwards-compat: also accept the old [{workspace_name: ...}] shape.
_raw_mon = CFG['monitored_workspaces']
MONITOR = [m if isinstance(m, str) else m['workspace_name'] for m in _raw_mon]
INGEST  = CFG['ingestion']
print(f'Observability workspace : {OBS_WS}')
print(f'Observability lakehouse : {OBS_LH}')
print(f'Monitored workspaces    : {MONITOR}')

In [ ]:
import requests, json
from datetime import datetime, timezone
from pyspark.sql import functions as F, Row

import notebookutils  # type: ignore
TOKEN = notebookutils.credentials.getToken(API['token_audience'])
HEADERS = {'Authorization': f'Bearer {TOKEN}'}
BASE = API['base_url'].rstrip('/')

def fab_get(path, params=None):
    r = requests.get(f'{BASE}{path}', headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def list_paged(path, params=None):
    items, ct = [], None
    while True:
        p = dict(params or {})
        if ct: p['continuationToken'] = ct
        data = fab_get(path, p)
        items.extend(data.get('value', []))
        ct = data.get('continuationToken')
        if not ct: break
    return items

## Resolve workspace names → IDs

In [ ]:
all_workspaces = list_paged('/workspaces')
ws_by_name = {w['displayName']: w for w in all_workspaces}

## Pull shortcuts from every lakehouse in each monitored workspace

In [ ]:
scanned_at = datetime.now(timezone.utc)
rows = []
for ws_name in MONITOR:
    ws = ws_by_name.get(ws_name)
    if not ws:
        print(f'  ! skipping workspace (not found / no access): {ws_name}')
        continue
    lakehouses = list_paged(f"/workspaces/{ws['id']}/items", {'type': 'Lakehouse'})
    print(f'  {ws_name}: {len(lakehouses)} lakehouse(s)')
    for lh in lakehouses:
        lh_name = lh['displayName']
        try:
            shortcuts = list_paged(f"/workspaces/{ws['id']}/items/{lh['id']}/shortcuts")
        except Exception as e:
            print(f'    ! {lh_name}: failed to list shortcuts: {e}')
            continue
        if not shortcuts:
            continue
        print(f'    {lh_name}: {len(shortcuts)} shortcut(s)')
        for sc in shortcuts:
            tgt = sc.get('target', {}) or {}
            ttype = tgt.get('type')
            ol = tgt.get('oneLake') or {}
            adls = tgt.get('adlsGen2') or {}
            s3 = tgt.get('amazonS3') or {}
            blob = tgt.get('azureBlobStorage') or {}
            gcs = tgt.get('googleCloudStorage') or {}
            ext = adls or s3 or blob or gcs
            rows.append({
                'scanned_at': scanned_at,
                'source_workspace_name': ws_name,
                'source_workspace_id': ws['id'],
                'source_item_name': lh_name,
                'source_item_id': lh['id'],
                'shortcut_name': sc.get('name'),
                'shortcut_path': sc.get('path'),
                'shortcut_full_path': f"{sc.get('path','').rstrip('/')}/{sc.get('name','')}",
                'target_type': ttype,
                'target_workspace_id': ol.get('workspaceId'),
                'target_item_id':      ol.get('itemId'),
                'target_path':         ol.get('path') or ext.get('subpath'),
                'target_connection_id': (ext or {}).get('connectionId'),
                'target_location':      (ext or {}).get('location'),
                'target_subpath':       (ext or {}).get('subpath'),
                'target_raw_json': json.dumps(tgt, default=str),
            })

print(f'Total shortcuts scanned: {len(rows)}')

## Overwrite dim_shortcut_map with current snapshot

Every run replaces the table with the latest set of shortcuts returned by the Fabric REST API. There is no history retained.

In [ ]:
from pyspark.sql.types import *

if not rows:
    print('No shortcuts found. dim_shortcut_map left unchanged.')
else:
    incoming = spark.createDataFrame(rows)
    (incoming.write.format('delta')
        .mode('overwrite').option('overwriteSchema','true')
        .saveAsTable(TBL['dim_shortcuts']))
    print(f"dim_shortcut_map overwritten. Row count: {spark.table(TBL['dim_shortcuts']).count()}")